In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_C.xpt"
csv_file = DATA_DIR / "ACQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched copy for fidelity comparison
source_df = df.copy()


print("=" * 70)
print("NHANES 2003-2004 ACQ_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD010A",
    "ACD010B",
    "ACD010C",
    "ACQ020",
    "ACQ030",
    "ACD040",
    "ACQ050",
    "ACQ060",
    "ACD070",
    "ACD080"
]


if df.columns.tolist() == EXPECTED_COLUMNS:

    print(
        "\nPASS: Column names/order match ACQ_C."
    )

else:

    print(
        "\nFAIL: Column structure differs."
    )

    print("Actual:")
    print(df.columns.tolist())


# ============================================================
# 4. CHECK KNOWN XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)

total_zero = int(
    zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{total_zero:,}"
)

print(
    "Tiny artifact values:",
    f"{total_tiny:,}"
)


if total_tiny > 0:

    print(
        "\nVariables containing tiny values:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    # IMPORTANT:
    # ACQ_C codebook does not require zero restoration
    # for the released variables in this file.
    #
    # Stop rather than silently changing anything.

    raise ValueError(
        "Unexpected tiny-value artifact found in ACQ_C. "
        "Do NOT automatically change it to zero."
    )


print(
    "PASS: No tiny zero artifact exists."
)


# ============================================================
# 5. CLEAN SEQN
# ============================================================

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 6. QUESTIONNAIRE CODE COLUMNS
# ============================================================

CODE_COLUMNS = [
    "ACD010A",
    "ACD010B",
    "ACD010C",
    "ACQ020",
    "ACQ030",
    "ACD040",
    "ACQ050",
    "ACQ060",
    "ACD070",
    "ACD080"
]


# ============================================================
# 7. VERIFY THERE ARE NO DECIMAL VALUES
# ============================================================

print("\n--- CODE VALUE FIDELITY ---")


for col in CODE_COLUMNS:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional.any():

        # Do NOT change the column if genuine
        # decimal values unexpectedly exist.

        raise ValueError(
            f"{col} contains fractional values. "
            "Column will NOT be converted."
        )


print(
    "PASS: All ACQ questionnaire values "
    "are whole-number source codes."
)


# ============================================================
# 8. FORMAT CODES WITHOUT .0
# ============================================================

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


# ============================================================
# 9. CDC ROW COUNT
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 7344:

    print(
        "PASS: Row count = 7,344"
    )

else:

    print(
        "FAIL: Unexpected row count:",
        len(df)
    )


if len(df.columns) == 11:

    print(
        "PASS: Released column count = 11"
    )

else:

    print(
        "FAIL: Column count =",
        len(df.columns)
    )


# ============================================================
# 10. CDC FREQUENCY CHECKS
# ============================================================

EXPECTED = {

    "ACD010A": {
        1: 5251,
        7: 1
    },

    "ACD010B": {
        2: 11
    },

    "ACD010C": {
        3: 314
    },

    "ACQ020": {
        1: 434,
        2: 285,
        3: 452,
        4: 416,
        5: 306,
        9: 2
    },

    "ACQ030": {
        1: 1046,
        2: 205,
        3: 204,
        4: 131,
        5: 307,
        9: 2
    },

    "ACD040": {
        1: 694,
        2: 198,
        3: 368,
        4: 202,
        5: 430,
        9: 3
    },

    "ACQ050": {
        1: 615,
        2: 134,
        3: 333,
        4: 196,
        5: 593,
        9: 24
    },

    "ACQ060": {
        1: 546,
        2: 167,
        3: 383,
        4: 241,
        5: 550,
        9: 8
    },

    "ACD070": {
        1: 507,
        2: 44,
        4: 1173,
        5: 119,
        10: 35,
        99: 17
    },

    "ACD080": {
        1: 574,
        2: 30,
        4: 1130,
        5: 123,
        10: 30,
        99: 8
    }
}


EXPECTED_MISSING = {

    "ACD010A": 2092,
    "ACD010B": 7333,
    "ACD010C": 7030,

    "ACQ020": 5449,
    "ACQ030": 5449,
    "ACD040": 5449,
    "ACQ050": 5449,
    "ACQ060": 5449,
    "ACD070": 5449,
    "ACD080": 5449
}


print("\n--- CDC CODE FREQUENCY CHECK ---")


for col in CODE_COLUMNS:

    passed = True

    expected_codes = EXPECTED[col]

    actual = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    if actual != expected_codes:

        passed = False

        print(
            f"FAIL: {col} code frequencies differ."
        )

        print(
            "Expected:",
            expected_codes
        )

        print(
            "Actual:",
            actual
        )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )

    expected_missing = (
        EXPECTED_MISSING[col]
    )


    if actual_missing != expected_missing:

        passed = False

        print(
            f"FAIL: {col} missing="
            f"{actual_missing}, "
            f"expected={expected_missing}"
        )


    if passed:

        print(
            f"PASS: {col}"
        )


# ============================================================
# 11. VERIFY NUMERIC VALUES WERE NOT CHANGED
# ============================================================

print("\n--- XPT VALUE FIDELITY BEFORE EXPORT ---")


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} value preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed."
        )


# ============================================================
# 12. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 13. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False
)


# ============================================================
# 14. XPT -> CSV EXACT VALUE CHECK
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


for col in source_df.columns:

    xpt_values = (
        source_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        xpt_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        print(
            f"FAIL: {col} differs from XPT."
        )


# ============================================================
# 15. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )


# ============================================================
# 16. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = csv_text.count(".0")


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    print(
        "CHECK: .0 formatting remains."
    )


# ============================================================
# 17. FINAL TINY VALUE CHECK
# ============================================================

numeric_csv = (
    csv_df.select_dtypes(
        include=[np.number]
    )
)


remaining_tiny = int(
    numeric_csv
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny artifact values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact exists."
    )


print("\n" + "=" * 70)
print("ACQ_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 ACQ_C XPT -> CSV
Rows: 7,344
Columns: 11

Variables:
['SEQN', 'ACD010A', 'ACD010B', 'ACD010C', 'ACQ020', 'ACQ030', 'ACD040', 'ACQ050', 'ACQ060', 'ACD070', 'ACD080']

PASS: Column names/order match ACQ_C.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact values: 0
PASS: No tiny zero artifact exists.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31125

--- CODE VALUE FIDELITY ---
PASS: All ACQ questionnaire values are whole-number source codes.

--- CDC STRUCTURE CHECK ---
PASS: Row count = 7,344
PASS: Released column count = 11

--- CDC CODE FREQUENCY CHECK ---
PASS: ACD010A
PASS: ACD010B
PASS: ACD010C
PASS: ACQ020
PASS: ACQ030
PASS: ACD040
PASS: ACQ050
PASS: ACQ060
PASS: ACD070
PASS: ACD080

--- XPT VALUE FIDELITY BEFORE EXPORT ---
PASS: SEQN       value preserved.
PASS: ACD010A    value preserved.
PASS: ACD010B    value preserved.
PASS: ACD010C    value preserved.
PASS: ACQ020     value preserved.
PASS: ACQ030     value preserved.
P